In [ ]:
import pandas as pd
import sys, os
sys.path.append(os.path.abspath(".."))
import config

In [ ]:
config.path_to_evaluation

In [ ]:
runs = ["run1", "run2", "run3", "run4", "run5"]
p_list = ["5000", "12000", "25000", "50000", "100000"]

Investigate cluster center features

In [ ]:
def format_value(mean_val, std_val):
    if abs(mean_val) >= 1e4 or (abs(mean_val) > 0 and abs(mean_val) < 1e-2):
        return f"{mean_val:.2e} ({std_val:.2e})"
    else:
        return f"{mean_val:.2f} ({std_val:.2f})"

In [ ]:
# cluster centers for div=0, average over 5 runs for each p
centers = []
for p in p_list:
    dataframes = []
    for run in runs:
        df = pd.read_parquet(f"{config.path_to_data}/{run}/cluster_centers/centers_{p}_0.parquet")[["erraticness","intermittency","abs_energy", "agg_autocorrelation_max", "agg_linear_trend_slope", "kurtosis", "mean", "median", "skewness","standard_deviation"]]
        ratio_row = {}
        for col in df.columns:
            val_0 = df.loc[0, col]
            val_1 = df.loc[1, col]
            
            # add a row for ratio of feature means
            if val_1 != 0:
                ratio_mean = val_0 / val_1
                ratio_row[col] = f"{ratio_mean:.2f}"
            else:
                ratio_row[col] = "inf"
        
        df.loc['ratio'] = ratio_row
        df = df.apply(pd.to_numeric, errors='coerce')
        dataframes.append(df)

    concatenated = pd.concat(dataframes, keys=range(len(dataframes)))

    # calculate mean and standard deviation of the cluster feature mean over the runs
    mean_df = concatenated.groupby(level=1).mean()
    std_df = concatenated.groupby(level=1).std()

    result_df = mean_df.copy()
    for col in mean_df.columns:
        result_df[col] = [
            format_value(mean_val, std_val)
            for mean_val, std_val in zip(mean_df[col], std_df[col])
        ]

    print(f"p:{p}", result_df)
    result_df.to_csv(f"{config.path_to_evaluation}/cluster_centers_avg/center_avg_{p}_0.csv")
    result_df.insert(0, "p", p)
    centers.append(result_df)
    
centers = pd.concat(centers)
centers.to_csv(f"{config.path_to_evaluation}/cluster_centers_avg/center_avgs_div0.csv")
centers

    